# Part 1.6 Checkpointers

After this notebook you can read a run's saved history by `thread_id` and resume a
failed run without repeating the steps that already finished.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

The graph saves its state after every step. When a step fails, the run picks up from the
last saved step instead of starting over.

<img src="../../assets/slides/langgraph-fault-tolerance.jpg" width="720">

## Build it

Run a claim on the finished graph, then list the checkpoints it left behind.
`claims_graph()` compiles it with an `InMemorySaver`.

In [ ]:
from kea.notebook import claim, claims_graph

graph = claims_graph()

config = {"configurable": {"thread_id": "claim-1001"}}
graph.invoke(claim("tear-01.jpg", "pax-1001", "KA201", 400), config)

snapshot = graph.get_state(config)
print("next node:", snapshot.next)
print("damage:", snapshot.values["assessment"]["damage_type"])
for state in reversed(list(graph.get_state_history(config))):
    print(state.metadata["step"], "next:", state.next)

Now add a checkpointer to your own graph. This one pays a claim, and the payments API is
down on the first try.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    claim_id: str
    payout_usd: float
    receipt: str


payments_up = False


def look_up(state: State) -> dict:
    print("look_up ran")
    return {"claim_id": "C-1"}


def pay(state: State) -> dict:
    if not payments_up:
        raise ConnectionError("payments API is down")
    return {"payout_usd": 100.0}


def notify(state: State) -> dict:
    return {"receipt": f"Paid ${state['payout_usd']:.2f} for {state['claim_id']}"}


builder = StateGraph(State)
builder.add_node("look_up", look_up)
builder.add_node("pay", pay)
builder.add_node("notify", notify)
builder.add_edge(START, "look_up")
builder.add_edge("look_up", "pay")
builder.add_edge("pay", "notify")
builder.add_edge("notify", END)
payout = builder.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "payout-1"}}
try:
    payout.invoke({}, config)
except ConnectionError as error:
    print("failed:", error)

print("next node:", payout.get_state(config).next)
print("saved state:", payout.get_state(config).values)

The saved state stops before `pay`. Bring the API back and resume the same thread.

In [ ]:
payments_up = True
result = payout.invoke(None, config)
print(result["receipt"])

## Under the hood

- The graph saves a **checkpoint after every step** (a "super-step"): the full state plus
  which node runs next. `next == ()` means the run finished.
- A **thread** is the sequence of checkpoints. Reuse a `thread_id` and the graph
  continues from the saved state. `invoke(None, config)` means "resume, no new input".
- `look_up` printed once. The resumed run started at `pay`, not at the beginning. For a
  claim, that means a failed `draft_letter` does not repeat the two vision calls.
- `InMemorySaver` forgets when the kernel stops. In production use a database
  checkpointer such as Postgres. `langgraph dev` and LangSmith Deployment supply one.
- Checkpoints are what make interrupts and time travel possible, which are next.

Docs: [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence),
[Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers).

## Recap

- Compile with a checkpointer and every run saves its history under a `thread_id`.
- `get_state` and `get_state_history` read it back.
- A failed run resumes from the last good step with `invoke(None, config)`.

**Next:** [Part 1.7 Interrupts](07_interrupts.ipynb)